# TensorForge encoder CV

Trains `intfloat/multilingual-e5-small` with category, secondary, and urgent heads.
Stage A (train → validation) is the number to quote. The 5-fold score is only a ranking check.

Before you run:

- Accelerator: **GPU T4**
- Internet: **On** (the notebook downloads the public encoder weights)
- Input: private dataset `tensorforge-rideeat-private`
- Do not paste an API key or upload `.env`

Use **Save Version → Save & Run All** so the job keeps running if you close the laptop.
When it finishes, download the `encoder` folder from the version output.


In [1]:
from pathlib import Path

matches = sorted(Path("/kaggle/input").rglob("train.csv"))
if len(matches) != 1:
    raise SystemExit(f"expected exactly one train.csv under /kaggle/input, found {matches}")
data_dir = matches[0].parent
print("data_dir", data_dir)
for name in ("train.csv", "validation.csv", "train.jsonl", "validation.jsonl"):
    path = data_dir / name
    print(
        f"{name}: {'ok' if path.is_file() else 'MISSING'} {path.stat().st_size if path.is_file() else 0}"
    )

data_dir /kaggle/input/datasets/uvarammohanaram/tensorforge-rideeat-private
train.csv: ok 2211821
validation.csv: ok 449668
train.jsonl: ok 2671652
validation.jsonl: ok 541582


In [2]:
from pathlib import Path

Path("app").mkdir(exist_ok=True)
Path("ml").mkdir(exist_ok=True)
print("package dirs ready")

package dirs ready


In [3]:
%%writefile app/__init__.py
"""TensorForge inference service."""


Writing app/__init__.py


In [4]:
%%writefile app/labels.py
"""Label enums and the fixed category -> team table.

Single source of truth for serving and training. tests/test_labels.py checks
these values against the OpenAPI spec.
"""

from __future__ import annotations

CATEGORIES: tuple[str, ...] = (
    "payment_refund",
    "ride_trip_issue",
    "lost_item",
    "order_missing_wrong",
    "delivery_delay",
    "food_quality",
    "account_promo",
    "safety_conduct",
    "app_technical",
    "general_inquiry",
    "spam_irrelevant",
)

TEAM_BY_CATEGORY: dict[str, str] = {
    "payment_refund": "Payments & Refunds",
    "ride_trip_issue": "Ride Operations",
    "lost_item": "Lost & Found",
    "order_missing_wrong": "Food Operations",
    "delivery_delay": "Delivery Operations",
    "food_quality": "Restaurant Quality",
    "account_promo": "Account Services",
    "safety_conduct": "Trust & Safety",
    "app_technical": "Tech Support",
    "general_inquiry": "Front-line Support",
    "spam_irrelevant": "Auto-close / Spam Filter",
}

CHANNELS: tuple[str, ...] = ("email", "chat", "call_transcript")

# The only secondary classes observed in the dataset (architecture section 5.1).
SECONDARY_CATEGORIES: tuple[str, ...] = (
    "ride_trip_issue",
    "payment_refund",
    "app_technical",
    "delivery_delay",
    "order_missing_wrong",
)

# These categories are never urgent in the labelled data. Spam is also a schema rule.
NEVER_URGENT: frozenset[str] = frozenset(
    {"account_promo", "app_technical", "general_inquiry", "spam_irrelevant"}
)


Writing app/labels.py


In [5]:
%%writefile app/text.py
"""Text normalization shared by training and serving.

Unicode NFC, keep ZWJ/ZWNJ (Sinhala and Tamil need them), strip other control
and format characters, collapse whitespace, lowercase Latin letters only.
"""

from __future__ import annotations

import re
import unicodedata

_WHITESPACE = re.compile(r"\s+")
# Cf characters we must not strip. ZWJ (U+200D) builds Sinhala conjuncts; ZWNJ (U+200C)
# is used in Tamil and Sinhala to block a conjunct.
_KEEP_FORMAT = frozenset("\u200c\u200d")


def _is_dropped(ch: str) -> bool:
    if ch in _KEEP_FORMAT:
        return False
    # Newlines and tabs are whitespace, not noise: they collapse to a space later.
    if ch in "\n\r\t":
        return False
    category = unicodedata.category(ch)
    return category in {"Cc", "Cf", "Cs"}


def _lower_latin(ch: str) -> str:
    if unicodedata.category(ch) != "Lu":
        return ch
    if unicodedata.name(ch, "").startswith("LATIN"):
        return ch.lower()
    return ch


def normalize(text: str) -> str:
    """Normalize one text field. Does not add the channel/subject wrapper."""
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    cleaned = "".join(_lower_latin(ch) for ch in text if not _is_dropped(ch))
    return _WHITESPACE.sub(" ", cleaned).strip()


def build_input(channel: str, subject: str | None, text: str) -> str:
    """Model input string. `ticket_id` and `language` are intentionally not arguments."""
    body = normalize(text)
    subj = normalize(subject or "")
    if subj:
        return f"[{channel}] {subj} || {body}"
    return f"[{channel}] || {body}"


Writing app/text.py


In [6]:
%%writefile app/inference.py
"""Classical inference engine and the post-processing rules.

The same `Engine.predict` path serves `/predict` and `/predict/batch` (and, later, jobs).
Encoder fusion is not part of the day-1 baseline.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np

from app.labels import NEVER_URGENT, SECONDARY_CATEGORIES, TEAM_BY_CATEGORY

ROOT = Path(__file__).resolve().parents[1]
DEFAULT_ARTIFACT = ROOT / "artifacts" / "classical.joblib"


def align_columns(proba: np.ndarray, classes: list[Any], labels: list[str]) -> np.ndarray:
    """Reorder `predict_proba` columns to `labels`. Missing classes stay 0."""
    proba = np.asarray(proba, dtype=np.float64)
    index = {str(classes[i]): i for i in range(len(classes))}
    out = np.zeros((proba.shape[0], len(labels)), dtype=np.float64)
    for column, label in enumerate(labels):
        source = index.get(str(label))
        if source is not None:
            out[:, column] = proba[:, source]
    return out


def positive_proba(model: Any, matrix: Any) -> np.ndarray:
    """P(class == 1) from a binary estimator. Works for bool and int labels."""
    proba = np.asarray(model.predict_proba(matrix), dtype=np.float64)
    for column, label in enumerate(model.classes_):
        if label == 1:
            return proba[:, column]
    return np.zeros(proba.shape[0], dtype=np.float64)


def apply_rules(
    category_proba: np.ndarray,
    category_labels: list[str],
    secondary_proba: np.ndarray,
    secondary_labels: list[str],
    urgent_proba: float,
    thresholds: dict[str, float],
) -> dict[str, Any]:
    """Turn one ticket's probabilities into a contract-safe prediction.

    Category is argmax. Secondary is the best of the five observed classes other
    than the primary, kept only when it clears the threshold. Urgency is forced
    off for the four categories that are never urgent, and spam also clears secondary.
    """
    category_index = int(np.argmax(category_proba))
    category = category_labels[category_index]
    confidence = float(np.clip(category_proba[category_index], 0.0, 1.0))
    confidence = float(round(confidence, 4))

    secondary: str | None = None
    if len(secondary_labels):
        candidates = [
            (float(secondary_proba[i]), secondary_labels[i])
            for i in range(len(secondary_labels))
            if secondary_labels[i] != category
        ]
        if candidates:
            best_p, best_label = max(candidates)
            if best_p >= thresholds["secondary"]:
                secondary = best_label

    is_urgent = float(urgent_proba) >= thresholds["urgent"]
    if category in NEVER_URGENT:
        is_urgent = False
    if category == "spam_irrelevant":
        secondary = None
        is_urgent = False
    if secondary == category:
        secondary = None

    return {
        "category": category,
        "secondary_category": secondary,
        "team": TEAM_BY_CATEGORY[category],
        "is_urgent": bool(is_urgent),
        "confidence": confidence,
        "needs_human_review": confidence < thresholds["review"],
    }


class Engine:
    def __init__(self, bundle: dict[str, Any]) -> None:
        self.model_version: str = str(bundle["model_version"])
        self.config_name: str = str(bundle.get("config_name", ""))
        self.vectorizer = bundle["vectorizer"]
        self.category_model = bundle["category_model"]
        self.category_labels: list[str] = list(bundle["category_labels"])
        self.secondary_models: dict[str, Any] = dict(bundle["secondary_models"])
        self.secondary_labels: list[str] = list(bundle.get("secondary_labels", SECONDARY_CATEGORIES))
        self.urgent_model = bundle["urgent_model"]
        self.thresholds: dict[str, float] = {
            "secondary": float(bundle["thresholds"]["secondary"]),
            "urgent": float(bundle["thresholds"]["urgent"]),
            "review": float(bundle["thresholds"]["review"]),
        }

    @classmethod
    def load(cls, path: Path | None = None) -> Engine:
        import joblib

        artifact = path or DEFAULT_ARTIFACT
        # Trusted artifact we trained and hashed at freeze time (S301 is our own joblib).
        bundle = joblib.load(artifact)  # noqa: S301
        return cls(bundle)

    def predict(self, ticket: dict[str, Any]) -> dict[str, Any]:
        return self.predict_many([ticket])[0]

    def predict_many(self, tickets: list[dict[str, Any]]) -> list[dict[str, Any]]:
        from app.text import build_input

        if not tickets:
            return []
        texts = [build_input(t["channel"], t.get("subject") or "", t["text"]) for t in tickets]
        matrix = self.vectorizer.transform(texts)
        category_proba = align_columns(
            self.category_model.predict_proba(matrix),
            list(self.category_model.classes_),
            self.category_labels,
        )
        secondary_columns = []
        for label in self.secondary_labels:
            model = self.secondary_models.get(label)
            if model is None:
                secondary_columns.append(np.zeros(len(tickets), dtype=np.float64))
            else:
                secondary_columns.append(positive_proba(model, matrix))
        secondary_proba = np.column_stack(secondary_columns)
        urgent = positive_proba(self.urgent_model, matrix)

        predictions: list[dict[str, Any]] = []
        for row, ticket in enumerate(tickets):
            pred = apply_rules(
                category_proba[row],
                self.category_labels,
                secondary_proba[row],
                self.secondary_labels,
                float(urgent[row]),
                self.thresholds,
            )
            pred["model_version"] = self.model_version
            if ticket.get("ticket_id") is not None:
                pred["ticket_id"] = ticket["ticket_id"]
            predictions.append(pred)
        return predictions


Writing app/inference.py


In [7]:
%%writefile ml/__init__.py
"""Training and evaluation package."""


Writing ml/__init__.py


In [8]:
%%writefile ml/data.py
"""Dataset loading and the fixed 5-fold split.

CSV is read with the csv module (text fields contain embedded newlines).
`folds.json` stores ticket ids only.
"""

from __future__ import annotations

import csv
import json
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data"
FOLDS_PATH = ROOT / "ml" / "folds.json"

FOLD_SEED = 42
N_SPLITS = 5


@dataclass(frozen=True, slots=True)
class TicketRow:
    ticket_id: str
    channel: str
    subject: str
    text: str
    language: str
    category: str
    secondary_category: str | None
    is_urgent: bool
    split: str


def _load_csv(path: Path, split: str) -> list[TicketRow]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Download the dataset into data/ (see data/README.md)."
        )
    rows: list[TicketRow] = []
    with path.open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for raw in reader:
            secondary = raw["secondary_category"] or None
            rows.append(
                TicketRow(
                    ticket_id=raw["ticket_id"],
                    channel=raw["channel"],
                    subject=raw["subject"],
                    text=raw["text"],
                    language=raw["language"],
                    category=raw["category"],
                    secondary_category=secondary,
                    is_urgent=raw["is_urgent"] == "true",
                    split=split,
                )
            )
    return rows


def load_split(split: str, data_dir: Path | None = None) -> list[TicketRow]:
    if split not in {"train", "validation"}:
        raise ValueError(f"unknown split {split!r}")
    directory = DATA_DIR if data_dir is None else data_dir
    return _load_csv(directory / f"{split}.csv", split)


def load_all(data_dir: Path | None = None) -> list[TicketRow]:
    """Train rows first, then validation. Order is stable."""
    return load_split("train", data_dir) + load_split("validation", data_dir)


def assign_folds(
    rows: list[TicketRow],
    n_splits: int = N_SPLITS,
    seed: int = FOLD_SEED,
) -> dict[str, int]:
    """Round-robin inside each category x language cell.

    Two cells have fewer than five tickets, so StratifiedKFold(n_splits=5) cannot
    be used on the joint key. Shuffling inside the cell and dealing round-robin
    still balances every cell across folds (counts differ by at most one).
    """
    groups: dict[tuple[str, str], list[str]] = defaultdict(list)
    for row in rows:
        groups[(row.category, row.language)].append(row.ticket_id)

    rng = np.random.default_rng(seed)
    fold_of: dict[str, int] = {}
    for key in sorted(groups):
        ids = groups[key][:]
        rng.shuffle(ids)
        for offset, ticket_id in enumerate(ids):
            fold_of[ticket_id] = offset % n_splits
    return fold_of


def write_folds(
    rows: list[TicketRow] | None = None,
    path: Path = FOLDS_PATH,
    n_splits: int = N_SPLITS,
    seed: int = FOLD_SEED,
) -> dict[str, int]:
    rows = load_all() if rows is None else rows
    fold_of = assign_folds(rows, n_splits=n_splits, seed=seed)
    payload = {
        "seed": seed,
        "n_splits": n_splits,
        "stratify": "category|language",
        "method": "round-robin within each category-language cell after a seeded shuffle",
        "fold_of": {ticket_id: fold_of[ticket_id] for ticket_id in sorted(fold_of)},
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    return fold_of


def load_folds(path: Path = FOLDS_PATH) -> dict[str, int]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    return {ticket_id: int(fold) for ticket_id, fold in payload["fold_of"].items()}


def main() -> None:
    rows = load_all()
    fold_of = write_folds(rows)
    sizes = [sum(1 for fold in fold_of.values() if fold == i) for i in range(N_SPLITS)]
    print(f"wrote {FOLDS_PATH} ({len(fold_of)} ids, fold sizes {sizes})")


if __name__ == "__main__":
    main()


Writing ml/data.py


In [9]:
%%writefile ml/evaluate.py
"""Metrics for category, secondary, urgency, calibration, and slices.

`append_experiment` writes one row to `ml/reports/experiments.csv`.
"""

from __future__ import annotations

import csv
from collections import Counter
from datetime import date
from pathlib import Path
from typing import Any

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

from app.labels import CATEGORIES, SECONDARY_CATEGORIES
from ml.data import TicketRow, load_all

ROOT = Path(__file__).resolve().parents[1]
REPORTS_DIR = ROOT / "ml" / "reports"
EXPERIMENTS_PATH = REPORTS_DIR / "experiments.csv"

CSV_COLUMNS = [
    "id",
    "date",
    "branch",
    "features",
    "hyperparameters",
    "stage_a_accuracy",
    "stage_a_macro_f1",
    "stage_a_urgent_f1",
    "stage_a_secondary_exact",
    "cv_accuracy",
    "cv_macro_f1",
    "cv_urgent_f1",
    "cv_secondary_exact",
    "notes",
]

METRIC_KEYS = (
    "n",
    "accuracy",
    "macro_f1",
    "per_class_f1",
    "per_class_support",
    "confusion_matrix",
    "secondary_exact",
    "secondary_per_class_f1",
    "urgent_precision",
    "urgent_recall",
    "urgent_f1",
    "ece",
    "brier",
    "urgent_brier",
    "by_language",
    "by_channel",
)


def _round(value: float) -> float:
    return round(float(value), 6)


def expected_calibration_error(confidence: np.ndarray, correct: np.ndarray, n_bins: int = 15) -> float:
    """ECE of the predicted class: bin confidence against accuracy."""
    confidence = np.asarray(confidence, dtype=np.float64)
    correct = np.asarray(correct, dtype=np.float64)
    total = len(confidence)
    if total == 0:
        return 0.0
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for index in range(n_bins):
        low, high = edges[index], edges[index + 1]
        if index == n_bins - 1:
            mask = (confidence >= low) & (confidence <= high)
        else:
            mask = (confidence >= low) & (confidence < high)
        count = int(mask.sum())
        if count == 0:
            continue
        ece += (count / total) * abs(float(correct[mask].mean()) - float(confidence[mask].mean()))
    return float(ece)


def multiclass_brier(proba: np.ndarray, gold: list[str], labels: list[str]) -> float:
    index = {label: i for i, label in enumerate(labels)}
    y_index = np.array([index[value] for value in gold])
    one_hot = np.zeros_like(proba, dtype=np.float64)
    one_hot[np.arange(len(gold)), y_index] = 1.0
    return float(np.mean(np.sum((proba - one_hot) ** 2, axis=1)))


def _binary_f1(gold: list[bool], pred: list[bool]) -> tuple[float, float, float]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        gold,
        pred,
        average="binary",
        pos_label=True,
        zero_division=0,
    )
    return _round(precision), _round(recall), _round(f1)


def _accuracy_f1(gold: list[str], pred: list[str], labels: list[str]) -> tuple[float, float]:
    if not gold:
        return 0.0, 0.0
    accuracy = accuracy_score(gold, pred)
    macro = f1_score(gold, pred, average="macro", labels=labels, zero_division=0)
    return _round(accuracy), _round(macro)


def _take(values: list[Any], indexes: list[int]) -> list[Any]:
    return [values[i] for i in indexes]


def compute_metrics(
    *,
    gold_category: list[str],
    pred_category: list[str],
    category_proba: np.ndarray,
    gold_secondary: list[str | None],
    pred_secondary: list[str | None],
    gold_urgent: list[bool],
    pred_urgent: list[bool],
    urgent_proba: np.ndarray,
    languages: list[str] | None = None,
    channels: list[str] | None = None,
    category_labels: list[str] | None = None,
) -> dict[str, Any]:
    """Full metric table. Probabilities are aligned to `category_labels`."""
    labels = list(category_labels or CATEGORIES)
    proba = np.asarray(category_proba, dtype=np.float64)
    confidence = proba.max(axis=1) if len(proba) else np.zeros(0)
    correct = np.array([gold == pred for gold, pred in zip(gold_category, pred_category, strict=True)])
    accuracy, macro_f1 = _accuracy_f1(gold_category, pred_category, labels)
    per_class = f1_score(gold_category, pred_category, average=None, labels=labels, zero_division=0)
    matrix = confusion_matrix(gold_category, pred_category, labels=labels)
    support = matrix.sum(axis=1)

    secondary_exact = 0.0
    if gold_secondary:
        matches = sum(gold == pred for gold, pred in zip(gold_secondary, pred_secondary, strict=True))
        secondary_exact = matches / len(gold_secondary)

    secondary_f1: dict[str, float] = {}
    for label in SECONDARY_CATEGORIES:
        gold_bin = [value == label for value in gold_secondary]
        pred_bin = [value == label for value in pred_secondary]
        secondary_f1[label] = _binary_f1(gold_bin, pred_bin)[2]

    urgent_precision, urgent_recall, urgent_f1 = _binary_f1(gold_urgent, pred_urgent)
    urgent_scores = np.asarray(urgent_proba, dtype=np.float64)
    urgent_brier = float(np.mean((urgent_scores - np.asarray(gold_urgent, dtype=np.float64)) ** 2))

    by_language: dict[str, dict[str, float | int]] = {}
    if languages is not None:
        for language in sorted(set(languages)):
            indexes = [i for i, value in enumerate(languages) if value == language]
            slice_acc, slice_f1 = _accuracy_f1(
                _take(gold_category, indexes),
                _take(pred_category, indexes),
                labels,
            )
            by_language[language] = {"n": len(indexes), "accuracy": slice_acc, "macro_f1": slice_f1}

    by_channel: dict[str, dict[str, float | int]] = {}
    if channels is not None:
        for channel in sorted(set(channels)):
            indexes = [i for i, value in enumerate(channels) if value == channel]
            slice_acc, slice_f1 = _accuracy_f1(
                _take(gold_category, indexes),
                _take(pred_category, indexes),
                labels,
            )
            by_channel[channel] = {"n": len(indexes), "accuracy": slice_acc, "macro_f1": slice_f1}

    return {
        "n": len(gold_category),
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "per_class_f1": {label: _round(score) for label, score in zip(labels, per_class, strict=True)},
        "per_class_support": {label: int(count) for label, count in zip(labels, support, strict=True)},
        "confusion_matrix": [[int(value) for value in row] for row in matrix.tolist()],
        "secondary_exact": _round(secondary_exact),
        "secondary_per_class_f1": secondary_f1,
        "urgent_precision": urgent_precision,
        "urgent_recall": urgent_recall,
        "urgent_f1": urgent_f1,
        "ece": _round(expected_calibration_error(confidence, correct.astype(np.float64))),
        "brier": _round(multiclass_brier(proba, gold_category, labels)),
        "urgent_brier": _round(urgent_brier),
        "by_language": by_language,
        "by_channel": by_channel,
    }


def majority_metrics(rows: list[TicketRow] | None = None) -> dict[str, Any]:
    """Train-set majority category, predicted for every validation ticket.

    Secondary is always null and urgency is always false. This is the floor.
    """
    rows = load_all() if rows is None else rows
    train = [row for row in rows if row.split == "train"]
    validation = [row for row in rows if row.split == "validation"]
    majority = Counter(row.category for row in train).most_common(1)[0][0]
    labels = list(CATEGORIES)
    column = labels.index(majority)
    proba = np.zeros((len(validation), len(labels)), dtype=np.float64)
    proba[:, column] = 1.0
    count = len(validation)
    metrics = compute_metrics(
        gold_category=[row.category for row in validation],
        pred_category=[majority] * count,
        category_proba=proba,
        gold_secondary=[row.secondary_category for row in validation],
        pred_secondary=[None] * count,
        gold_urgent=[row.is_urgent for row in validation],
        pred_urgent=[False] * count,
        urgent_proba=np.zeros(count, dtype=np.float64),
        languages=[row.language for row in validation],
        channels=[row.channel for row in validation],
    )
    metrics["majority_category"] = majority
    return metrics


def append_experiment(row: dict[str, Any], path: Path = EXPERIMENTS_PATH) -> None:
    """Append one experiment. Creates the file and header when missing."""
    path.parent.mkdir(parents=True, exist_ok=True)
    new_file = not path.is_file() or path.stat().st_size == 0
    with path.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS, extrasaction="ignore")
        if new_file:
            writer.writeheader()
        payload = {column: row.get(column, "") for column in CSV_COLUMNS}
        if not payload["date"]:
            payload["date"] = date.today().isoformat()
        for column in CSV_COLUMNS:
            value = payload[column]
            if value is None:
                payload[column] = ""
            elif isinstance(value, float):
                payload[column] = f"{value:.6f}"
        writer.writerow(payload)


def record_majority(path: Path = EXPERIMENTS_PATH) -> dict[str, Any]:
    metrics = majority_metrics()
    append_experiment(
        {
            "id": "majority_class",
            "branch": "baseline",
            "features": "none",
            "hyperparameters": f"predict {metrics['majority_category']} for every ticket",
            "stage_a_accuracy": metrics["accuracy"],
            "stage_a_macro_f1": metrics["macro_f1"],
            "stage_a_urgent_f1": metrics["urgent_f1"],
            "stage_a_secondary_exact": metrics["secondary_exact"],
            "notes": "floor: train majority category, secondary null, urgent false, scored on validation",
        },
        path=path,
    )
    return metrics


Writing ml/evaluate.py


In [10]:
%%writefile ml/truncation.py
"""Head-and-tail token window shared by encoder training and, later, serving.

A 256-token budget keeps the start of the ticket (channel, subject, opening) and
the end (the ask is often there). The middle is what gets dropped on long texts.
"""

from __future__ import annotations

MAX_LENGTH = 256
# Content tokens only. CLS and SEP use the other two positions (190 + 64 + 2 = 256).
HEAD_TOKENS = 190
TAIL_TOKENS = 64


def head_tail_window(token_ids: list[int], content_budget: int, head: int, tail: int) -> list[int]:
    """Keep the first `head` and last `tail` ids when the sequence exceeds the budget."""
    if content_budget <= 0:
        return []
    if len(token_ids) <= content_budget:
        return list(token_ids)
    head_n = min(max(head, 0), content_budget)
    tail_n = min(max(tail, 0), content_budget - head_n)
    if tail_n <= 0:
        return list(token_ids[:content_budget])
    return list(token_ids[:head_n]) + list(token_ids[-tail_n:])


Writing ml/truncation.py


In [11]:
%%writefile ml/train_encoder.py
"""Fine-tune a multilingual encoder with category, secondary, and urgent heads.

Device-agnostic: the laptop can smoke-test it, and the same command runs 5-fold
CV on a Kaggle T4. Stage A (train to validation) is the number to quote. The
pooled out-of-fold score is only a ranking check, same caveat as the classical
models, because near-duplicate templates cross fold boundaries.

    python -m ml.train_encoder --data-dir data --out-dir ml/runs/encoder_e5_small

Checkpoints are the best epoch per split (macro-F1 on that split's held-out
rows). A finished split is skipped on the next launch, so a dead Kaggle session
can be re-run without repeating completed folds.
"""

from __future__ import annotations

import argparse
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from app.inference import apply_rules
from app.labels import CATEGORIES, SECONDARY_CATEGORIES
from app.text import build_input
from ml.data import TicketRow, assign_folds, load_all, load_folds
from ml.evaluate import EXPERIMENTS_PATH, append_experiment, compute_metrics
from ml.truncation import HEAD_TOKENS, MAX_LENGTH, TAIL_TOKENS, head_tail_window

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

DEFAULT_MODEL = "intfloat/multilingual-e5-small"
THRESHOLDS = {"secondary": 0.5, "urgent": 0.5, "review": 0.5}
LOSS_SECONDARY = 0.5
LOSS_URGENT = 0.5


@dataclass(frozen=True)
class RunConfig:
    model_name: str
    epochs: int
    batch_size: int
    lr: float
    patience: int
    seed: int
    smoke: bool


class MultiHeadEncoder(nn.Module):
    """Shared encoder, mean-pooled, then three task heads."""

    def __init__(self, encoder: nn.Module) -> None:
        super().__init__()
        self.encoder = encoder
        hidden = int(encoder.config.hidden_size)
        self.dropout = nn.Dropout(0.1)
        self.category = nn.Linear(hidden, len(CATEGORIES))
        self.secondary = nn.Linear(hidden, len(SECONDARY_CATEGORIES))
        self.urgent = nn.Linear(hidden, 1)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        pooled = self.dropout(pooled)
        urgent = self.urgent(pooled).squeeze(-1)
        return self.category(pooled), self.secondary(pooled), urgent


class EncodedTickets(Dataset):
    def __init__(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        category: torch.Tensor,
        secondary: torch.Tensor,
        urgent: torch.Tensor,
    ) -> None:
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.category = category
        self.secondary = secondary
        self.urgent = urgent

    def __len__(self) -> int:
        return int(self.category.shape[0])

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        return (
            self.input_ids[index],
            self.attention_mask[index],
            self.category[index],
            self.secondary[index],
            self.urgent[index],
        )


def _seed_everything(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _encode_texts(tokenizer: Any, texts: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    if cls_id is None or sep_id is None:
        raise RuntimeError("tokenizer has no CLS/SEP ids; refusing a silent truncation change")
    content_budget = MAX_LENGTH - 2
    input_ids = torch.full((len(texts), MAX_LENGTH), int(pad_id), dtype=torch.long)
    attention_mask = torch.zeros((len(texts), MAX_LENGTH), dtype=torch.long)
    for row, text in enumerate(texts):
        raw = tokenizer.encode(text, add_special_tokens=False)
        window = head_tail_window(raw, content_budget, HEAD_TOKENS, TAIL_TOKENS)
        ids = [int(cls_id), *window, int(sep_id)]
        width = len(ids)
        input_ids[row, :width] = torch.tensor(ids, dtype=torch.long)
        attention_mask[row, :width] = 1
    return input_ids, attention_mask


def _label_tensors(rows: list[TicketRow]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    category_index = {label: index for index, label in enumerate(CATEGORIES)}
    secondary_index = {label: index for index, label in enumerate(SECONDARY_CATEGORIES)}
    category = torch.tensor([category_index[row.category] for row in rows], dtype=torch.long)
    secondary = torch.zeros((len(rows), len(SECONDARY_CATEGORIES)), dtype=torch.float32)
    for row_index, row in enumerate(rows):
        label = row.secondary_category
        if label in secondary_index:
            secondary[row_index, secondary_index[label]] = 1.0
    urgent = torch.tensor([1.0 if row.is_urgent else 0.0 for row in rows], dtype=torch.float32)
    return category, secondary, urgent


def _class_weights(category: torch.Tensor) -> torch.Tensor:
    counts = torch.bincount(category, minlength=len(CATEGORIES)).float()
    weights = counts.sum() / counts.clamp(min=1.0)
    return weights / weights.mean()


def _pos_weight(binary: torch.Tensor) -> torch.Tensor:
    if binary.ndim == 1:
        positive = binary.sum()
        negative = binary.numel() - positive
        return (negative / positive.clamp(min=1.0)).reshape(1)
    positive = binary.sum(dim=0)
    negative = binary.shape[0] - positive
    return negative / positive.clamp(min=1.0)


def _texts(rows: list[TicketRow]) -> list[str]:
    return [build_input(row.channel, row.subject, row.text) for row in rows]


def _predictions(
    category_proba: np.ndarray,
    secondary_proba: np.ndarray,
    urgent_proba: np.ndarray,
) -> list[dict[str, Any]]:
    labels = list(CATEGORIES)
    secondary_labels = list(SECONDARY_CATEGORIES)
    predictions: list[dict[str, Any]] = []
    for index in range(category_proba.shape[0]):
        predictions.append(
            apply_rules(
                category_proba[index],
                labels,
                secondary_proba[index],
                secondary_labels,
                float(urgent_proba[index]),
                THRESHOLDS,
            )
        )
    return predictions


def _score(rows: list[TicketRow], predictions: list[dict[str, Any]], category_proba: np.ndarray, urgent_proba: np.ndarray) -> dict[str, Any]:
    return compute_metrics(
        gold_category=[row.category for row in rows],
        pred_category=[item["category"] for item in predictions],
        category_proba=category_proba,
        gold_secondary=[row.secondary_category for row in rows],
        pred_secondary=[item["secondary_category"] for item in predictions],
        gold_urgent=[row.is_urgent for row in rows],
        pred_urgent=[bool(item["is_urgent"]) for item in predictions],
        urgent_proba=urgent_proba,
        languages=[row.language for row in rows],
        channels=[row.channel for row in rows],
    )


def _jsonable(metrics: dict[str, Any]) -> dict[str, Any]:
    """Drop nothing; metrics are already plain numbers, lists, and dicts."""
    return metrics


@torch.no_grad()
def _predict(
    model: MultiHeadEncoder,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    batch_size: int,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    use_amp = device.type == "cuda"
    category_parts: list[np.ndarray] = []
    secondary_parts: list[np.ndarray] = []
    urgent_parts: list[np.ndarray] = []
    for start in range(0, input_ids.shape[0], batch_size):
        stop = start + batch_size
        ids = input_ids[start:stop].to(device, non_blocking=True)
        mask = attention_mask[start:stop].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            category_logits, secondary_logits, urgent_logits = model(ids, mask)
        category_parts.append(torch.softmax(category_logits.float(), dim=-1).cpu().numpy())
        secondary_parts.append(torch.sigmoid(secondary_logits.float()).cpu().numpy())
        urgent_parts.append(torch.sigmoid(urgent_logits.float()).cpu().numpy())
    return (
        np.concatenate(category_parts, axis=0),
        np.concatenate(secondary_parts, axis=0),
        np.concatenate(urgent_parts, axis=0),
    )


def _train_one(
    train_rows: list[TicketRow],
    eval_rows: list[TicketRow],
    tokenizer: Any,
    encoder: nn.Module,
    config: RunConfig,
    directory: Path,
    device: torch.device,
) -> dict[str, Any]:
    directory.mkdir(parents=True, exist_ok=True)
    metrics_path = directory / "metrics.json"
    proba_path = directory / "proba.npz"
    if metrics_path.is_file() and proba_path.is_file():
        print(f"skip {directory.name}: checkpoint already finished", flush=True)
        saved = json.loads(metrics_path.read_text(encoding="utf-8"))
        return saved

    _seed_everything(config.seed)
    train_ids, train_mask = _encode_texts(tokenizer, _texts(train_rows))
    eval_ids, eval_mask = _encode_texts(tokenizer, _texts(eval_rows))
    train_cat, train_sec, train_urg = _label_tensors(train_rows)
    dataset = EncodedTickets(train_ids, train_mask, train_cat, train_sec, train_urg)
    generator = torch.Generator()
    generator.manual_seed(config.seed)
    loader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=device.type == "cuda",
    )

    model = MultiHeadEncoder(encoder).to(device)
    category_weight = _class_weights(train_cat).to(device)
    secondary_pos = _pos_weight(train_sec).to(device)
    urgent_pos = _pos_weight(train_urg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=0.01)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_f1 = -1.0
    best_epoch = 0
    stale = 0
    best_state: dict[str, torch.Tensor] | None = None

    for epoch in range(1, config.epochs + 1):
        model.train()
        running = 0.0
        seen = 0
        for ids, mask, cat_y, sec_y, urg_y in loader:
            ids = ids.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            cat_y = cat_y.to(device, non_blocking=True)
            sec_y = sec_y.to(device, non_blocking=True)
            urg_y = urg_y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                cat_logits, sec_logits, urg_logits = model(ids, mask)
            loss_cat = nn.functional.cross_entropy(cat_logits.float(), cat_y, weight=category_weight)
            loss_sec = nn.functional.binary_cross_entropy_with_logits(
                sec_logits.float(), sec_y, pos_weight=secondary_pos
            )
            loss_urg = nn.functional.binary_cross_entropy_with_logits(
                urg_logits.float().unsqueeze(1),
                urg_y.unsqueeze(1),
                pos_weight=urgent_pos,
            )
            loss = loss_cat + LOSS_SECONDARY * loss_sec + LOSS_URGENT * loss_urg
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            running += float(loss.item()) * ids.shape[0]
            seen += int(ids.shape[0])

        category_proba, secondary_proba, urgent_proba = _predict(
            model, eval_ids, eval_mask, config.batch_size * 2, device
        )
        metrics = _score(eval_rows, _predictions(category_proba, secondary_proba, urgent_proba), category_proba, urgent_proba)
        print(
            f"{directory.name} epoch {epoch} loss {running / max(seen, 1):.4f} "
            f"macro_f1 {metrics['macro_f1']:.4f} acc {metrics['accuracy']:.4f} "
            f"urgent_f1 {metrics['urgent_f1']:.4f}",
            flush=True,
        )
        if metrics["macro_f1"] > best_f1:
            best_f1 = float(metrics["macro_f1"])
            best_epoch = epoch
            stale = 0
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        else:
            stale += 1
            if stale >= config.patience:
                print(f"{directory.name} early stop at epoch {epoch}", flush=True)
                break

    if best_state is None:
        raise RuntimeError(f"no epoch completed for {directory}")
    model.load_state_dict(best_state)
    category_proba, secondary_proba, urgent_proba = _predict(
        model, eval_ids, eval_mask, config.batch_size * 2, device
    )
    predictions = _predictions(category_proba, secondary_proba, urgent_proba)
    metrics = _score(eval_rows, predictions, category_proba, urgent_proba)
    metrics["best_epoch"] = best_epoch
    model.cpu()
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    torch.save(best_state, directory / "model.pt")
    np.savez_compressed(
        proba_path,
        ticket_id=np.array([row.ticket_id for row in eval_rows]),
        category_proba=category_proba,
        secondary_proba=secondary_proba,
        urgent_proba=urgent_proba,
    )
    metrics_path.write_text(json.dumps(_jsonable(metrics), indent=2) + "\n", encoding="utf-8")
    return metrics


def _load_encoder(model_name: str) -> tuple[Any, nn.Module]:
    from transformers import AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    encoder = AutoModel.from_pretrained(model_name)
    return tokenizer, encoder


def _fresh_encoder(model_name: str) -> nn.Module:
    from transformers import AutoModel

    return AutoModel.from_pretrained(model_name)


def _subsample(rows: list[TicketRow], limit: int, seed: int) -> list[TicketRow]:
    if len(rows) <= limit:
        return rows
    rng = np.random.default_rng(seed)
    picked = rng.choice(len(rows), size=limit, replace=False)
    return [rows[int(index)] for index in sorted(picked)]


def _headline(metrics: dict[str, Any]) -> dict[str, float]:
    return {
        "accuracy": float(metrics["accuracy"]),
        "macro_f1": float(metrics["macro_f1"]),
        "urgent_f1": float(metrics["urgent_f1"]),
        "secondary_exact": float(metrics["secondary_exact"]),
    }


def _pool_fold_metrics(rows: list[TicketRow], out_dir: Path, n_splits: int) -> dict[str, Any]:
    by_id = {row.ticket_id: row for row in rows}
    order = [row.ticket_id for row in rows]
    category = np.zeros((len(rows), len(CATEGORIES)), dtype=np.float64)
    secondary = np.zeros((len(rows), len(SECONDARY_CATEGORIES)), dtype=np.float64)
    urgent = np.zeros(len(rows), dtype=np.float64)
    filled = np.zeros(len(rows), dtype=bool)
    position = {ticket_id: index for index, ticket_id in enumerate(order)}
    for fold in range(n_splits):
        payload = np.load(out_dir / f"fold_{fold}" / "proba.npz", allow_pickle=False)
        for row_index, ticket_id in enumerate(payload["ticket_id"].tolist()):
            slot = position[str(ticket_id)]
            category[slot] = payload["category_proba"][row_index]
            secondary[slot] = payload["secondary_proba"][row_index]
            urgent[slot] = payload["urgent_proba"][row_index]
            filled[slot] = True
    if not bool(filled.all()):
        missing = int((~filled).sum())
        raise RuntimeError(f"{missing} tickets have no out-of-fold prediction")
    heldout = [by_id[ticket_id] for ticket_id in order]
    predictions = _predictions(category, secondary, urgent)
    return _score(heldout, predictions, category, urgent)


def run(data_dir: Path, out_dir: Path, config: RunConfig, skip_stage_a: bool, skip_cv: bool) -> dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    device = _device()
    print(f"device {device.type} model {config.model_name}", flush=True)
    log_experiment = not (out_dir / "experiment.json").is_file()
    rows = load_all(data_dir)
    if config.smoke:
        train_rows = _subsample([row for row in rows if row.split == "train"], 128, config.seed)
        eval_rows = _subsample([row for row in rows if row.split == "validation"], 32, config.seed)
        rows = train_rows + eval_rows
    train_rows = [row for row in rows if row.split == "train"]
    valid_rows = [row for row in rows if row.split == "validation"]

    tokenizer, template = _load_encoder(config.model_name)
    summary: dict[str, Any] = {
        "model_name": config.model_name,
        "max_length": MAX_LENGTH,
        "head_tokens": HEAD_TOKENS,
        "tail_tokens": TAIL_TOKENS,
        "thresholds": THRESHOLDS,
        "smoke": config.smoke,
    }
    (out_dir / "run_config.json").write_text(
        json.dumps(
            {
                "model_name": config.model_name,
                "epochs": config.epochs,
                "batch_size": config.batch_size,
                "lr": config.lr,
                "patience": config.patience,
                "seed": config.seed,
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    stage_a: dict[str, Any] | None = None
    if not skip_stage_a:
        # Each split needs its own weight init. Reuse the already-downloaded object for stage A.
        stage_a = _train_one(
            train_rows,
            valid_rows,
            tokenizer,
            template,
            config,
            out_dir / "stage_a",
            device,
        )
        summary["stage_a"] = _headline(stage_a)
        print(
            "stage A "
            f"macro_f1 {summary['stage_a']['macro_f1']:.4f} "
            f"acc {summary['stage_a']['accuracy']:.4f} "
            f"urgent_f1 {summary['stage_a']['urgent_f1']:.4f}",
            flush=True,
        )
    del template

    cv: dict[str, Any] | None = None
    if not skip_cv and not config.smoke:
        folds_path = Path(__file__).resolve().parent / "folds.json"
        fold_of = load_folds(folds_path) if folds_path.is_file() else assign_folds(rows)
        n_splits = 5
        for fold in range(n_splits):
            held_in = [row for row in rows if fold_of[row.ticket_id] != fold]
            held_out = [row for row in rows if fold_of[row.ticket_id] == fold]
            _train_one(
                held_in,
                held_out,
                tokenizer,
                _fresh_encoder(config.model_name),
                RunConfig(
                    model_name=config.model_name,
                    epochs=config.epochs,
                    batch_size=config.batch_size,
                    lr=config.lr,
                    patience=config.patience,
                    seed=config.seed + fold,
                    smoke=False,
                ),
                out_dir / f"fold_{fold}",
                device,
            )
        cv = _pool_fold_metrics(rows, out_dir, n_splits)
        summary["cv"] = _headline(cv)
        (out_dir / "cv_metrics.json").write_text(json.dumps(cv, indent=2) + "\n", encoding="utf-8")
        print(
            "cv "
            f"macro_f1 {summary['cv']['macro_f1']:.4f} "
            f"acc {summary['cv']['accuracy']:.4f} "
            f"urgent_f1 {summary['cv']['urgent_f1']:.4f}",
            flush=True,
        )

    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    if log_experiment and not config.smoke and (stage_a is not None or cv is not None):
        row = {
            "id": "encoder_e5_small",
            "branch": "encoder",
            "features": "multilingual-e5-small mean-pool head+tail 256",
            "hyperparameters": (
                f"AdamW lr={config.lr} epochs<={config.epochs} patience={config.patience} "
                f"batch={config.batch_size} class_weight+pos_weight fp16 thresholds 0.5"
            ),
            "notes": "Stage A is the number to quote. CV/OOF is a ranking check only.",
        }
        if stage_a is not None:
            row["stage_a_accuracy"] = stage_a["accuracy"]
            row["stage_a_macro_f1"] = stage_a["macro_f1"]
            row["stage_a_urgent_f1"] = stage_a["urgent_f1"]
            row["stage_a_secondary_exact"] = stage_a["secondary_exact"]
        if cv is not None:
            row["cv_accuracy"] = cv["accuracy"]
            row["cv_macro_f1"] = cv["macro_f1"]
            row["cv_urgent_f1"] = cv["urgent_f1"]
            row["cv_secondary_exact"] = cv["secondary_exact"]
        if EXPERIMENTS_PATH.parent.is_dir():
            append_experiment(row)
        (out_dir / "experiment.json").write_text(json.dumps(row, indent=2) + "\n", encoding="utf-8")
    return summary


def main() -> None:
    parser = argparse.ArgumentParser(description="Train the multilingual encoder heads.")
    parser.add_argument("--data-dir", type=Path, default=Path("data"))
    parser.add_argument("--out-dir", type=Path, default=Path("ml/runs/encoder_e5_small"))
    parser.add_argument("--model", default=DEFAULT_MODEL)
    parser.add_argument("--epochs", type=int, default=6)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--patience", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--smoke", action="store_true", help="One short stage-A run on a few rows.")
    parser.add_argument("--skip-stage-a", action="store_true")
    parser.add_argument("--skip-cv", action="store_true")
    args = parser.parse_args()
    out_dir = args.out_dir
    if args.smoke and out_dir == Path("ml/runs/encoder_e5_small"):
        out_dir = Path("ml/runs/encoder_smoke")
    epochs = 1 if args.smoke else args.epochs
    batch_size = 8 if args.smoke else args.batch_size
    config = RunConfig(
        model_name=args.model,
        epochs=epochs,
        batch_size=batch_size,
        lr=args.lr,
        patience=1 if args.smoke else args.patience,
        seed=args.seed,
        smoke=args.smoke,
    )
    run(
        args.data_dir,
        out_dir,
        config,
        skip_stage_a=args.skip_stage_a,
        skip_cv=True if args.smoke else args.skip_cv,
    )


if __name__ == "__main__":
    main()


Writing ml/train_encoder.py


In [12]:
import os
import subprocess
import sys

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working"
env["TOKENIZERS_PARALLELISM"] = "false"
env["HF_HUB_DISABLE_TELEMETRY"] = "1"
command = [
    sys.executable,
    "-m",
    "ml.train_encoder",
    "--data-dir",
    str(data_dir),
    "--out-dir",
    "/kaggle/working/encoder",
    "--epochs",
    "6",
    "--batch-size",
    "16",
    "--patience",
    "2",
]
print(" ".join(command), flush=True)
subprocess.check_call(command, cwd="/kaggle/working", env=env)
print("finished", flush=True)
print("download /kaggle/working/encoder (summary.json, stage_a/, fold_0..4/)", flush=True)

/usr/bin/python3 -m ml.train_encoder --data-dir /kaggle/input/datasets/uvarammohanaram/tensorforge-rideeat-private --out-dir /kaggle/working/encoder --epochs 6 --batch-size 16 --patience 2
device cuda model intfloat/multilingual-e5-small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2736.64it/s]


stage_a epoch 1 loss 3.1402 macro_f1 0.5633 acc 0.5637 urgent_f1 0.6393
stage_a epoch 2 loss 1.9065 macro_f1 0.7725 acc 0.7775 urgent_f1 0.8427
stage_a epoch 3 loss 1.3329 macro_f1 0.7631 acc 0.7625 urgent_f1 0.8941
stage_a epoch 4 loss 1.0117 macro_f1 0.7855 acc 0.7825 urgent_f1 0.9080
stage_a epoch 5 loss 0.7892 macro_f1 0.7740 acc 0.7688 urgent_f1 0.9059
stage_a epoch 6 loss 0.6113 macro_f1 0.7773 acc 0.7750 urgent_f1 0.9240
stage_a early stop at epoch 6
stage A macro_f1 0.7855 acc 0.7825 urgent_f1 0.9080


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2735.28it/s]


fold_0 epoch 1 loss 3.2337 macro_f1 0.7000 acc 0.7068 urgent_f1 0.7731
fold_0 epoch 2 loss 2.0235 macro_f1 0.9117 acc 0.9171 urgent_f1 0.7668
fold_0 epoch 3 loss 1.4296 macro_f1 0.9803 acc 0.9828 urgent_f1 0.8362
fold_0 epoch 4 loss 1.0889 macro_f1 0.9927 acc 0.9909 urgent_f1 0.8940
fold_0 epoch 5 loss 0.8526 macro_f1 0.9926 acc 0.9929 urgent_f1 0.8739
fold_0 epoch 6 loss 0.6765 macro_f1 0.9904 acc 0.9909 urgent_f1 0.9108
fold_0 early stop at epoch 6


...

fold_2 epoch 1 loss 3.2792 macro_f1 0.7124 acc 0.7149 urgent_f1 0.6138
fold_2 epoch 2 loss 2.0929 macro_f1 0.9368 acc 0.9386 urgent_f1 0.8571
fold_2 epoch 3 loss 1.4894 macro_f1 0.9706 acc 0.9719 urgent_f1 0.9406
fold_2 epoch 4 loss 1.1287 macro_f1 0.9897 acc 0.9906 urgent_f1 0.9314
fold_2 epoch 5 loss 0.8860 macro_f1 0.9914 acc 0.9938 urgent_f1 0.9694
fold_2 epoch 6 loss 0.7000 macro_f1 0.9931 acc 0.9948 urgent_f1 0.9458


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2801.86it/s]


fold_3 epoch 1 loss 3.3174 macro_f1 0.5888 acc 0.6072 urgent_f1 0.5621
fold_3 epoch 2 loss 2.1555 macro_f1 0.8943 acc 0.8976 urgent_f1 0.8333
fold_3 epoch 3 loss 1.5184 macro_f1 0.9700 acc 0.9725 urgent_f1 0.9187
fold_3 epoch 4 loss 1.1276 macro_f1 0.9738 acc 0.9778 urgent_f1 0.9187
fold_3 epoch 5 loss 0.8853 macro_f1 0.9886 acc 0.9905 urgent_f1 0.9648
fold_3 epoch 6 loss 0.7027 macro_f1 0.9894 acc 0.9916 urgent_f1 0.9845


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2693.85it/s]


fold_4 epoch 1 loss 3.3183 macro_f1 0.6729 acc 0.6942 urgent_f1 0.4718
fold_4 epoch 2 loss 2.1438 macro_f1 0.8937 acc 0.8938 urgent_f1 0.9151
fold_4 epoch 3 loss 1.5090 macro_f1 0.9770 acc 0.9785 urgent_f1 0.9238
fold_4 epoch 4 loss 1.1454 macro_f1 0.9915 acc 0.9914 urgent_f1 0.9238
fold_4 epoch 5 loss 0.9045 macro_f1 0.9917 acc 0.9914 urgent_f1 0.9798
fold_4 epoch 6 loss 0.7137 macro_f1 0.9986 acc 0.9979 urgent_f1 0.9700
cv macro_f1 0.9927 acc 0.9933 urgent_f1 0.9541
finished
download /kaggle/working/encoder (summary.json, stage_a/, fold_0..4/)
